# Decision Tree Classifier — Interpretable Non-Linear Benchmark

**Project:** Encrypted DNS & QUIC Traffic Intelligence  
**Focus:** Observable Traffic Flow Classification without Payload Decryption  
**Model Family:** `Decision Tree`  
**Zero-Leakage Policy:** Strictly relies on observable packet lengths and inter-arrival timing. Zero IP, port, SNI, or payload content leakage.

---

### Model Overview
Transparent single-tree baseline measuring the degree of separability and interpretable hierarchical decision paths on raw timing and packet sizes.

**Protocol Target Classes (6):**
- `DOH`: DNS-over-HTTPS (RFC 8484)
- `DOQ`: DNS-over-QUIC (RFC 9250)
- `DOH3`: DNS-over-HTTP/3 (RFC 9114)
- `QUIC`: Raw QUIC Transport Protocol (RFC 9000)
- `HTTPS_WEB`: Standard HTTPS Web Browsing
- `HTTP3_WEB`: Modern Web Browsing over HTTP/3 / QUIC
*(DOT protocol completely removed as per specification)*


In [ ]:
# ── 1. Environment Setup & Core Dependencies ──
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)
import joblib

from sklearn.tree import DecisionTreeClassifier, plot_tree

# Visualization aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'Helvetica'
plt.rcParams['axes.edgecolor'] = '#cbd5e1'
plt.rcParams['axes.linewidth'] = 0.8

RANDOM_STATE = 42
MODEL_NAME = "Decision Tree"
print(f"Environment initialized for {MODEL_NAME}.")


## 2. Dataset Ingestion & Validation

We load the consolidated multi-vantage point dataset from `../PROJECT_DATASET/`.  
By default, this notebook ingests `FULL_COMBINED_WITH_METADATA.csv` (or `FULL_COMBINED.csv`), covering captures from:
- **Prayagraj (Airtel AS45609 & Reliance Jio AS55836)**
- **Tumsar (Maharashtra Airtel/Docker Lab)**
- **Mirzapur (Reliance Jio)**


In [ ]:
# ── 2. Load Consolidated Multi-Vantage Point Dataset ──
DATA_DIR = Path("../PROJECT_DATASET")
DATASET_PATH = DATA_DIR / "FULL_COMBINED_WITH_METADATA.csv"

# Fallback to standard 5-col version if metadata file is not present
if not DATASET_PATH.exists():
    DATASET_PATH = DATA_DIR / "FULL_COMBINED.csv"

print(f"Loading dataset from: {DATASET_PATH}")
df = pd.read_csv(DATASET_PATH)
print(f"Dataset successfully loaded: {df.shape[0]:,d} total records, {df.shape[1]} columns.")

# Audit class distribution
print("\nClass Distribution:")
print(df['traffic_class'].value_counts())

# Audit missing values
null_count = df.isnull().sum().sum()
assert null_count == 0, f"Found {null_count} missing values in dataset!"
print("\nData Integrity Check: PASS (0 missing values, zero DOT entries).")


## 3. Feature Matrix Construction & Zero-Leakage Policy

The ML features strictly follow the kinematic packet observable schema:
1. `Current packet size` ($L_i$)
2. `Previous packet size` ($L_{i-1}$)
3. `Current time gap` ($\Delta t_i = t_i - t_{i-1}$)
4. `Previous time gap` ($\Delta t_{i-1} = t_{i-1} - t_{i-2}$)


In [ ]:
# ── 3. Feature Matrix & Target Label Vector ──
FEATURE_COLS = [
    'Current packet size',
    'Previous packet size',
    'Current time gap',
    'Previous time gap'
]

CLASS_MAP = {
    'DOH': 0,
    'DOQ': 1,
    'DOH3': 2,
    'QUIC': 3,
    'HTTPS_WEB': 4,
    'HTTP3_WEB': 5
}
CLASS_NAMES = {v: k for k, v in CLASS_MAP.items()}

X = df[FEATURE_COLS].values
y = df['traffic_class'].map(CLASS_MAP).values

print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape:  {y.shape}")
print(f"Classes present: {[CLASS_NAMES[i] for i in sorted(np.unique(y))]}")


## 4. Stratified Train / Validation / Test Splitting

To ensure robust evaluation:
- **Train Set (70%):** Model parameter estimation.
- **Validation Set (15%):** Hyperparameter tuning and model checkpointing.
- **Holdout Test Set (15%):** Completely unseen test set for unbiased final reporting.


In [ ]:
# ── 4. Stratified 70 / 15 / 15 Partitioning ──
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=RANDOM_STATE
)

# Further split train_val into 70% train and 15% val (relative 15/85 ≈ 0.1765)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=(0.15 / 0.85), stratify=y_train_val, random_state=RANDOM_STATE
)

print(f"Train set:      {X_train.shape[0]:,d} samples ({X_train.shape[0]/len(X):.1%})")
print(f"Validation set: {X_val.shape[0]:,d} samples ({X_val.shape[0]/len(X):.1%})")
print(f"Test set:       {X_test.shape[0]:,d} samples ({X_test.shape[0]/len(X):.1%})")


## 5. Model Architecture & Pipeline Construction

We configure `Decision Tree`.
Tree-based models do not require monotonic feature scaling; raw packet dimensions are directly processed.


In [ ]:
# ── 5. Define Model Instance ──
# Decision Tree with entropy splitting criterion and controlled depth
model = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=12,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=RANDOM_STATE
)

pipeline = model
print("Model instance constructed successfully:")
print(pipeline)


## 6. Model Training & 5-Fold Stratified Cross-Validation

We evaluate model stability using 5-fold cross-validation on the training set, followed by full fitting.


In [ ]:
# ── 6. 5-Fold Stratified Cross-Validation & Fit ──
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ['accuracy', 'f1_macro', 'f1_weighted']

print(f"Running 5-fold cross-validation for {MODEL_NAME}...")
start_time = time.time()
cv_results = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
train_time = time.time() - start_time

print(f"\n5-Fold Cross-Validation Results ({train_time:.2f}s):")
print(f"  Accuracy:    {np.mean(cv_results['test_accuracy']):.4f} ± {np.std(cv_results['test_accuracy']):.4f}")
print(f"  Macro-F1:    {np.mean(cv_results['test_f1_macro']):.4f} ± {np.std(cv_results['test_f1_macro']):.4f}")
print(f"  Weighted-F1: {np.mean(cv_results['test_f1_weighted']):.4f} ± {np.std(cv_results['test_f1_weighted']):.4f}")

# Fit on entire training set
print(f"\nFitting {MODEL_NAME} on full training split ({len(X_train):,d} samples)...")
pipeline.fit(X_train, y_train)
print("Fit complete.")


## 7. Validation Set Evaluation

Evaluating performance on the independent validation split ($15\%$) to verify generalization before unlocking the test set.


In [ ]:
# ── 7. Evaluate on Validation Set ──
y_val_pred = pipeline.predict(X_val)

val_acc = accuracy_score(y_val, y_val_pred)
val_f1_macro = f1_score(y_val, y_val_pred, average='macro')
val_f1_weighted = f1_score(y_val, y_val_pred, average='weighted')

print(f"Validation Accuracy:    {val_acc:.4f}")
print(f"Validation Macro-F1:    {val_f1_macro:.4f}")
print(f"Validation Weighted-F1: {val_f1_weighted:.4f}")


## 8. Final Evaluation on Unseen Test Set

Testing the trained model on the holdout test set ($15\%$) and computing detailed per-class classification metrics.


In [ ]:
# ── 8. Test Set Evaluation & Metrics ──
start_inf = time.time()
y_test_pred = pipeline.predict(X_test)
inf_time_ms = (time.time() - start_inf) / len(X_test) * 1000

test_acc = accuracy_score(y_test, y_test_pred)
test_f1_macro = f1_score(y_test, y_test_pred, average='macro')
test_f1_weighted = f1_score(y_test, y_test_pred, average='weighted')

print(f"============================================================")
print(f"  {MODEL_NAME} — FINAL TEST SET PERFORMANCE")
print(f"============================================================")
print(f"  Test Accuracy:     {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"  Macro-F1 Score:    {test_f1_macro:.4f}")
print(f"  Weighted-F1:       {test_f1_weighted:.4f}")
print(f"  Inference Latency: {inf_time_ms:.4f} ms / packet vector")
print(f"============================================================\n")

target_names = [CLASS_NAMES[i] for i in sorted(np.unique(y_test))]
print("Detailed Classification Report:")
print(classification_report(y_test, y_test_pred, target_names=target_names, digits=4))


## 9. Visual Confusion Matrix

Heatmap visualizing predicted vs. actual protocol classes to inspect false-positive and false-negative distributions.


In [ ]:
# ── 9. Confusion Matrix Heatmap ──
cm = confusion_matrix(y_test, y_test_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(
    cm_norm, annot=True, fmt='.2%', cmap='Blues',
    xticklabels=target_names, yticklabels=target_names,
    cbar=True, ax=ax, linewidths=0.5
)
ax.set_title(f'{MODEL_NAME} — Test Normalized Confusion Matrix', fontweight='bold', fontsize=13)
ax.set_xlabel('Predicted Label', fontweight='bold', labelpad=10)
ax.set_ylabel('Actual Label', fontweight='bold', labelpad=10)
plt.tight_layout()
plt.show()


## 10. Model-Specific Deep Dive: Decision Tree Analysis

Analyzing the internal representation and mechanics of `Decision Tree`.


In [ ]:
# ── Decision Tree Structure & Pruning Insights ──
print(f'Tree Depth: {model.get_depth()}')
print(f'Total Leaf Nodes: {model.get_n_leaves()}')

plt.figure(figsize=(18, 8))
plot_tree(model, max_depth=3, feature_names=FEATURE_COLS, class_names=list(CLASS_NAMES.values()), filled=True, rounded=True, fontsize=9)
plt.title(f'{MODEL_NAME} — Top Decision Splits (Depth ≤ 3)', fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Cross-Network Robustness Check (Airtel vs. Jio Domain Shift)

Evaluating model robustness across different networks and geographical vantage points.


In [ ]:
# ── 11. Cross-Network Domain Shift Evaluation ──
if 'isp' in df.columns:
    airtel_mask = (df['isp'] == 'Airtel')
    jio_mask = (df['isp'] == 'Jio')

    if airtel_mask.sum() > 0 and jio_mask.sum() > 0:
        X_airtel = df.loc[airtel_mask, FEATURE_COLS].values
        y_airtel = df.loc[airtel_mask, 'traffic_class'].map(CLASS_MAP).values

        X_jio = df.loc[jio_mask, FEATURE_COLS].values
        y_jio = df.loc[jio_mask, 'traffic_class'].map(CLASS_MAP).values

        # Train on Airtel -> Test on Jio
        print(f"Training on Airtel ({len(X_airtel):,d}) -> Testing on Jio ({len(X_jio):,d})...")
        pipeline.fit(X_airtel, y_airtel)
        jio_pred = pipeline.predict(X_jio)
        acc_a2j = accuracy_score(y_jio, jio_pred)
        f1_a2j = f1_score(y_jio, jio_pred, average='macro')
        print(f"  Cross-Net (Airtel -> Jio) Accuracy: {acc_a2j:.4f} | Macro-F1: {f1_a2j:.4f}")
else:
    print("Dataset does not contain 'isp' metadata column; skipping cross-network test.")


## 12. Model Serialization & Export

Saving the trained pipeline for downstream evaluation, inference server deployment, or reproducibility.


In [ ]:
# ── 12. Save Trained Model to Disk ──
MODELS_DIR = Path("../PROJECT_NOTEBOOKS/saved_models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

save_path = MODELS_DIR / f"03_Decision_Tree.joblib"
joblib.dump(pipeline, save_path)
print(f"Trained model pipeline successfully serialized to: {save_path.resolve()}")
